**Questão 1**

Alterar os nomes das colunas

In [5]:
import numpy as np
import pandas as pd
df = pd.read_csv('saude_do_sono_estilo_vida.csv')

In [6]:
df.rename(columns={
    'ID': 'Identificador',
    'Pressão sanguíneaaaa':'Pressão sanguínea',
    'Ocupação': 'Profissão',
    'Categoria BMI': 'IMC'
}, inplace=True)

In [299]:
print(df.columns)

Index(['Identificador', 'Gênero', 'Idade', 'Profissão', 'Duração do sono',
       'Qualidade do sono', 'Nível de atividade física', 'Nível de estresse',
       'IMC', 'Pressão sanguínea', 'Frequência cardíaca', 'Passos diários',
       'Distúrbio do sono'],
      dtype='object')


**Questão 3:** Cálculo de Impacto de Obesidade em Eng. Software

Das pessoas que atuam com engenharia de software qual a porcentagem de obesos?

In [300]:
# Filtra os dados de acordo com a profissao solicitada
df_eng_software = df[df['Profissão'] == 'Eng. de Software']
df_eng_software

,Identificador,Gênero,Idade,Profissão,Duração do sono,Qualidade do sono,Nível de atividade física,Nível de estresse,IMC,Pressão sanguínea,Frequência cardíaca,Passos diários,Distúrbio do sono
0,1,Homem,27,Eng. de Software,6.1,6,42,6,Sobrepeso,126/83,77,4200,Nenhuma
5,6,Homem,28,Eng. de Software,5.9,4,30,8,Obesidade,140/90,85,3000,Insônia
84,85,Homem,35,Eng. de Software,7.5,8,60,5,Peso normal,120/80,70,8000,Nenhuma
92,93,Homem,35,Eng. de Software,7.5,8,60,5,Peso normal,120/80,70,8000,Nenhuma


In [301]:
# Faz a contagem de profissionais de cada categoria de imc
cont_imc_eng = df_eng_software.groupby('Profissão')['IMC'].value_counts()

# Calcula e imprime o total de profissionais
total_imc_eng = cont_imc_eng.sum()

profissao_eng = df_eng_software['Profissão'].values[0]
print(f"Total de {profissao_eng} da Base: {total_imc_eng}")

Total de Eng. de Software da Base: 4


In [302]:
# Constroi a tabela resumo
# precisa usar reset para transformar em df
df_imc_eng = cont_imc_eng.reset_index(name='Total')
df_imc_eng['Total %'] = df_imc_eng['Total'] * 100 / total_imc_eng # calcula %
df_imc_eng


,Profissão,IMC,Total,Total %
0,Eng. de Software,Peso normal,2,50.0
1,Eng. de Software,Obesidade,1,25.0
2,Eng. de Software,Sobrepeso,1,25.0


In [312]:
# A partir daqui, e adicional, porque o que ja foi feito antes responde

# Funcao que retorna o total da categoria de IMC
def retorna_imc(grupo, categoria, num_format=False):
  # checa se a categoria nao existe nos valores de IMC
  if categoria not in grupo['IMC'].values:
    return categoria, 0 # se nao existe, o valor e zero

  # Atribui padrao para pegar o valor com porcentagem na coluna de total
  nome_coluna = 'Total %'

  if num_format: # Se formato existir
    nome_coluna = 'Total' # Pega valor numerico na coluna de total

  # Retorna categoria e seu total no padrao perc/num
  return categoria, grupo.loc[grupo['IMC'] == categoria, nome_coluna].values[0]

# Atribui os valores retornados da funcao chamada
imc_cat_print, obesos_perc_print = retorna_imc(df_imc_eng, 'Obesidade')

# Imprime resposta
print(f"Resposta: {profissao_eng} com {imc_cat_print} corresponde a {obesos_perc_print:.1f}%")


Resposta: Eng. de Software com Obesidade corresponde a 25.0%


**Questão 4:** Comparativo de Sono: Advocacia ou Vendas vs. média geral

De acordo com os dados, advogar ou ser representante de vendas faz você dormir menos? (Use o método ‘isin’, considere a média)

In [304]:
# Retorna a media geral de sono da base
media_sono = df['Duração do sono'].mean()
media_sono

np.float64(7.129490616621984)

In [305]:
# Filtra os dados de acordo com as profissoes solicitadas
filtro_profissao = df['Profissão'].isin(['Advogado(a)', 'Representante de Vendas'])
df_adv_vendas = df[filtro_profissao]
df_adv_vendas.head()

,Identificador,Gênero,Idade,Profissão,Duração do sono,Qualidade do sono,Nível de atividade física,Nível de estresse,IMC,Pressão sanguínea,Frequência cardíaca,Passos diários,Distúrbio do sono
3,4,Homem,28,Representante de Vendas,5.9,4,30,8,Obesidade,140/90,85,3000,Apneia do sono
4,5,Homem,28,Representante de Vendas,5.9,4,30,8,Obesidade,140/90,85,3000,Apneia do sono
93,94,Homem,35,Advogado(a),7.4,7,60,5,Obesidade,135/88,84,3300,Apneia do sono
109,110,Homem,37,Advogado(a),7.4,8,60,5,Normal,130/85,68,8000,Nenhuma
111,112,Homem,37,Advogado(a),7.4,8,60,5,Normal,130/85,68,8000,Nenhuma


In [306]:
# Imprime a media geral para comparar
print(f"Média Geral de Sono da Base: {media_sono:.2f} horas")

# Traz a media de cada profissao
media_sono_adv_vendas = df_adv_vendas.groupby('Profissão')['Duração do sono'].agg('mean')
media_sono_adv_vendas

Média Geral de Sono da Base: 7.13 horas


,Duração do sono
Profissão,
Advogado(a),7.410638
Representante de Vendas,5.900000


In [307]:
# A partir daqui, e adicional, porque o que ja foi feito antes responde

# Funcao que compara as medias do grupo com a media geral da base
def comparar_media_sono(medias_grupo, media_geral=media_sono):
  # Percorre cada profissao e sua media no grupo de medias
  for profissao, media_grupo in medias_grupo.items():
    # Checa se a media do grupo e menor que a media geral
    if media_grupo < media_geral:
      return profissao, media_grupo
  return

# Atribui os valores retornados da funcao chamada
profissao_print, media_print = comparar_media_sono(media_sono_adv_vendas)

# Imprime resposta
print(f"Resposta: Ser {profissao_print} faz você dormir menos (Média: {media_print:.2f}h)")


Resposta: Ser Representante de Vendas faz você dormir menos (Média: 5.90h)


**Questão 5**

Entre quem fez enfermagem e quem fez medicina, quem tem menos horas de sono?

In [8]:
# Cria um subconjunto apenas com as profissões de interesse
enf_med = ['Enfermeiro(a)','Médico(a)']
saude = df[df['Profissão'].isin(enf_med)]

# Calcula a média de duração do sono para cada profissão
media = saude.groupby('Profissão')['Duração do sono'].mean().reset_index()

# Ordena o resultado
media = media.sort_values(by='Duração do sono')

# Exibe o resultado para cada profissão
print(media)
print('\n')

# Identifica o profissional com menos hora de sono
menos = media['Profissão'].iloc[0]

# Exibe o resultado
print('O profissional com menos hora de sono é o(a)', menos)

print('\n')

       Profissão  Duração do sono
1      Médico(a)         6.970423
0  Enfermeiro(a)         7.048611


O profissional com menos hora de sono é o(a) Médico(a)




**Questão 6**

Criar subconjuntos com as colunas: Identificador, gênero, idade, pressão sanguínea e frequência cardíaca

In [12]:
subconjunto = df[
    [
        'Identificador',
        'Gênero',
        'Idade',
        'Pressão sanguínea',
        'Frequência cardíaca'
    ]
]

In [13]:
print(subconjunto.head())

   Identificador Gênero  Idade Pressão sanguínea  Frequência cardíaca
0              1  Homem     27            126/83                   77
1              2  Homem     28            125/80                   75
2              3  Homem     28            125/80                   75
3              4  Homem     28            140/90                   85
4              5  Homem     28            140/90                   85


In [14]:
subconjunto.head()

,Identificador,Gênero,Idade,Pressão sanguínea,Frequência cardíaca
0,1,Homem,27,126/83,77
1,2,Homem,28,125/80,75
2,3,Homem,28,125/80,75
3,4,Homem,28,140/90,85
4,5,Homem,28,140/90,85


**Questão 7**

Descubra qual a profissão menos frequente no conjunto.

In [11]:
# Calcula a ocorrência de cada profissão (em porcentagem)
df_frequencia = df['Profissão'].value_counts(normalize=True)*100

# Exibe a ocorrência de cada profissão (em porcentagem)
print(df_frequencia)
print('\n')

# Identifica a profissão com menor ocorrência
menos_freq = df_frequencia.index[-1]

# Exibe o resultado
print('O profissional menos frequente na amostra de entrevistados é o(a)', menos_freq)

print('\n')

Profissão
Enfermeiro(a)              19.302949
Médico(a)                  19.034853
Engenheiro(a)              16.890080
Advogado(a)                12.600536
Professor(a)               10.723861
Contador(a)                 9.919571
Pessoa Vendendora           8.579088
Eng. de Software            1.072386
Cientista                   1.072386
Representante de Vendas     0.536193
Gerente                     0.268097
Name: proportion, dtype: float64


O profissional menos frequente na amostra de entrevistados é o(a) Gerente


